## SCORE: 0.87248

## 1. Установка необходимых библиотек

In [ ]:
!pip install --upgrade featuretools >> None

## 2. Импорт библиотек и настройка окружения

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold

from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import early_stopping as lgb_early_stopping

import warnings
warnings.filterwarnings("ignore")

## 3. Загрузка данных

In [ ]:
transaction_file = '/kaggle/input/dataset-generated/dataset_generated_with_cats.csv'
dataset_generated_with_cats = pd.read_csv(transaction_file)

In [ ]:
target_file = '/kaggle/input/alfa-challenge/train.pa'
df_target = pd.read_parquet(target_file)

In [ ]:
print("Данные успешно загружены.")

## 4. Подготовка данных

In [ ]:
target = df_target[['client_num', 'target']]

data_for_model = dataset_generated_with_cats.merge(target, on='client_num', how='inner')
print("Признаки и целевая переменная объединены для модели.")

cat_features = data_for_model.select_dtypes(include=['object', 'category']).columns.tolist()

X = data_for_model.drop(['client_num', 'target'], axis=1)
y = data_for_model['target']

### Подготовка данных для теста

In [ ]:
all_client_nums = dataset_generated_with_cats['client_num']
train_client_nums = df_target['client_num']

test_client_nums = all_client_nums[~all_client_nums.isin(train_client_nums)].reset_index(drop=True)

test_features = dataset_generated_with_cats[dataset_generated_with_cats['client_num'].isin(test_client_nums)].reset_index(drop=True)
X_test = test_features.drop(['client_num'], axis=1)

## 5. Расчет весов классов

In [ ]:
unique_classes = np.sort(y.unique())
class_counts = y.value_counts()
total_samples = len(y)
class_weights = {c: total_samples / (len(unique_classes) * class_counts[c]) for c in unique_classes}
print("Веса классов рассчитаны.")

def map_class_weights(y_labels, class_weights):
    return y_labels.map(class_weights).values

weights = map_class_weights(y, class_weights)
print("Веса для выборки рассчитаны.")

## 6. Определение кастомной метрики WMAE

In [ ]:
class WMAEMetric:
    def get_final_error(self, error, weight):
        return error / weight

    def is_max_optimal(self):
        return False

    def evaluate(self, approxes, target, weight):
        approx = approxes[0]
        target = np.array(target)
        weight = np.ones_like(target) if weight is None else np.array(weight)
        error = np.sum(weight * np.abs(target - approx))
        return error, np.sum(weight)
print("Кастомная метрика WMAE определена.")

## 7. Инициализация параметров для кросс-валидации и предсказаний

In [ ]:
n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

oof_predictions = np.zeros((X.shape[0], 3))
test_predictions = np.zeros((X_test.shape[0], 3))

## 8. Кодирование категориальных признаков

In [ ]:
X_encoded = X.copy()
X_test_encoded = X_test.copy()

for col in cat_features:
    X_encoded[col] = X_encoded[col].astype('category').cat.codes
    X_test_encoded[col] = X_test_encoded[col].astype('category').cat.codes

## 9. Подготовка параметров моделей

In [ ]:
catboost_params = {
    'iterations': 1000,
    'eval_metric': WMAEMetric(),
    'loss_function': 'MAE',
    'random_seed': 42,
    'early_stopping_rounds': 50,
    'use_best_model': True
}

lgb_params = {
    'n_estimators': 500,
    'objective': 'mae',
    'random_state': 42
}

xgb_params = {
    'n_estimators': 500,
    'objective': 'reg:squarederror',
    'random_state': 42,
    'eval_metric': 'mae'
}

## 9. Обучение моделей и стекинг

In [ ]:
for fold, (train_idx, valid_idx) in enumerate(kf.split(X_encoded, y)):
    print(f"\nFold {fold + 1}/{n_folds}")
    X_train_fold, X_valid_fold = X_encoded.iloc[train_idx], X_encoded.iloc[valid_idx]
    y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]
    weights_train_fold, weights_valid_fold = weights[train_idx], weights[valid_idx]

    # CatBoost
    train_pool = Pool(
        data=X.iloc[train_idx],
        label=y_train_fold, 
        weight=weights_train_fold, 
        cat_features=cat_features
    )
    valid_pool = Pool(
        data=X.iloc[valid_idx],
        label=y_valid_fold, 
        weight=weights_valid_fold, 
        cat_features=cat_features
    )

    model_cb = CatBoostRegressor(**catboost_params)
    model_cb.fit(
        train_pool,
        eval_set=valid_pool
    )
    oof_predictions[valid_idx, 0] = model_cb.predict(X.iloc[valid_idx])
    test_predictions[:, 0] += model_cb.predict(X_test) / n_folds

    # LightGBM
    model_lgb = LGBMRegressor(**lgb_params)
    model_lgb.fit(
        X_train_fold, y_train_fold,
        sample_weight=weights_train_fold,
        eval_set=[(X_valid_fold, y_valid_fold)],
        eval_sample_weight=[weights_valid_fold],
        callbacks=[lgb_early_stopping(50, first_metric_only=True)]
    )
    oof_predictions[valid_idx, 1] = model_lgb.predict(X_valid_fold)
    test_predictions[:, 1] += model_lgb.predict(X_test_encoded) / n_folds

    # XGBoost
    model_xgb = XGBRegressor(**xgb_params)
    model_xgb.fit(
        X_train_fold, y_train_fold,
        sample_weight=weights_train_fold,
        eval_set=[(X_valid_fold, y_valid_fold)],
        early_stopping_rounds=50
    )
    oof_predictions[valid_idx, 2] = model_xgb.predict(X_valid_fold)
    test_predictions[:, 2] += model_xgb.predict(X_test_encoded) / n_folds

## 10. Обучение метамодели

In [ ]:
meta_features = oof_predictions
meta_target = y

meta_model = RandomForestRegressor(n_estimators=100, random_state=42)
meta_model.fit(meta_features, meta_target)
print("Метамодель обучена.")

In [ ]:
y_pred_meta = meta_model.predict(meta_features)

wmae_meta = np.sum(weights * np.abs(y - y_pred_meta)) / np.sum(weights)
print('WMAE на обучающей выборке для метамодели:', wmae_meta)

## 11. Финальное предсказание и сохранение результатов

In [ ]:
meta_test_features = test_predictions
test_pred_meta = meta_model.predict(meta_test_features)
test_pred_meta_rounded = np.round(test_pred_meta).astype(int)

submission = pd.DataFrame({
    'client_num': test_features['client_num'],
    'target': test_pred_meta_rounded
})

submission.to_csv('submission.csv', index=False)
print("Предсказания сохранены в файл 'submission.csv'.")